# AI-Based Resume Screening System

The objective of this project is to develop an AI-based system that analyzes resumes and compares them with a given job description. The system extracts key skills from the resume, matches them with required job skills, calculates a matching score, and provides an explanation based on the level of match.

Install 

In [1]:
!pip install transformers langchain-core

import Libraries

In [2]:
import sys
!{sys.executable} -m pip install transformers langchain-core langchain
!{sys.executable} -m pip install torch --index-url https://download.pytorch.org/whl/cpu

Looking in indexes: https://download.pytorch.org/whl/cpu


In [3]:
from transformers import pipeline
from langchain_core.prompts import PromptTemplate

Load Local Model 

In [4]:
llm = pipeline("text-generation", model="distilgpt2")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Resume and Job Description

In [5]:
resume = """
John Doe
Skills: Python, Machine Learning, SQL, Pandas, NumPy, Deep Learning
Experience: 2 years in Data Science projects including model building and data analysis
Tools: Jupyter, Scikit-learn, TensorFlow
"""

job_description = """
Looking for a Data Scientist with:
Python, Machine Learning, SQL, Pandas, NumPy
Experience in data analysis and model building
"""

Extract Function

In [6]:
def extract_info(resume_text):
    lines = resume_text.split("\n")
    
    skills = ""
    experience = ""
    tools = ""
    
    for line in lines:
        if "Skills:" in line:
            skills = line.replace("Skills:", "").strip()
        elif "Experience:" in line:
            experience = line.replace("Experience:", "").strip()
        elif "Tools:" in line:
            tools = line.replace("Tools:", "").strip()
    
    return {
        "skills": skills,
        "experience": experience,
        "tools": tools
    }

Score Function 

In [7]:
def calculate_score(resume_data, job_description):
    jd_skills = ["Python", "Machine Learning", "SQL", "Pandas", "NumPy"]
    
    resume_skills = [skill.strip() for skill in resume_data["skills"].split(",")]
    
    match_count = 0
    
    for skill in jd_skills:
        if skill in resume_skills:
            match_count += 1
    
    score = (match_count / len(jd_skills)) * 100
    
    return score, match_count

Extract Data

In [8]:
data = extract_info(resume)

Calculate Score

In [9]:
score, match_count = calculate_score(data, job_description)

LLM Explanation

In [10]:
simple_prompt = f"""
Candidate Score: {score}

Skills: {data['skills']}

Job requires: Python, Machine Learning, SQL, Pandas, NumPy

Explain in 2 lines why this score is given.
"""

def generate_explanation(score, match_count):
    if score == 100:
        return "Candidate perfectly matches all required skills."
    elif score >= 70:
        return f"Candidate matches most required skills ({match_count} matched)."
    elif score >= 40:
        return f"Candidate matches some skills ({match_count} matched)."
    else:
        return "Candidate does not match required skills."

explanation = generate_explanation(score, match_count)

print("Matching Score:", score)
print("Explanation:", explanation)

Matching Score: 100.0
Explanation: Candidate perfectly matches all required skills.
